## K-means on the Full Feature Sampled Dataset

#### Load Libraries

In [15]:
import pandas as pd
import os
import time
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from scipy.stats import mode
import numpy as np


os.chdir("../")

#### Trying K-means on full dataset

In [16]:
# Read file in 
df = pd.read_csv("data/raw/HIGGS.csv", header = None)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/HIGGS.csv'

In [ ]:
df.columns = ['label',
                     'lepton_pT', 'lepton_eta', 'lepton_phi',
                     'missing_E_mag', 'missing_E_phi',
                     'jet1_pT', 'jet1_eta', 'jet1_phi', 'jet1_btag',
                     'jet2_pT', 'jet2_eta', 'jet2_phi', 'jet2_btag',
                     'jet3_pT', 'jet3_eta', 'jet3_phi', 'jet3_btag',
                     'jet4_pT', 'jet4_eta', 'jet4_phi', 'jet4_btag',
                     'm_jj', 'm_jjj', 'm_lv', 'm_jlv',
                     'm_bb', 'm_wbb', 'm_wwbb']

In [ ]:
rows_before_na = len(df)
df = df.dropna()
rows_missing_data = rows_before_na - len(df)

print(f"Dropped {rows_missing_data} rows with missing values.")

rows_before_duplicates = len(df)
df = df.drop_duplicates()
rows_duplicate_data = rows_before_duplicates - len(df)

print(f"Dropped {rows_duplicate_data} duplicated rows.")

print(f"Remaining Number of Rows: {len(df)}")

Dropped 0 rows with missing values.
Dropped 278698 duplicated rows.
Remaining Number of Rows: 10721302


In [ ]:
X = df.drop(columns="label")
y = df["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
start_time = time.time()

kmeans = KMeans(n_clusters=2, init = 'k-means++', random_state = 42, n_init='auto') 

kmeans.fit(X_scaled)

end_time = time.time()

training_time = end_time - start_time
print(f"K-means runtime: {training_time:.2f} seconds")
print(f"Iterations: {kmeans.n_iter_}")

start_time = time.time()

y_kmeans = kmeans.predict(X_scaled)

end_time = time.time()
prediction_time = end_time - start_time

print(f"K-means prediction time: {prediction_time:.2f} seconds")

K-means runtime: 16.07 seconds
Iterations: 18
K-means prediction time: 2.50 seconds


In [ ]:
mapped_labels = np.zeros_like(y_kmeans) # same length as y_kmeans

for i in range(2):
    bool_mask = (y_kmeans==i)
    mapped_labels[bool_mask] = mode(y[bool_mask], keepdims=True).mode[0]

accuracy = accuracy_score(y, mapped_labels)*100
print(f"The accuracy of the K-means algorithm on the full feature dataset is {accuracy}%.")


The accuracy of the K-means algorithm on the full feature dataset is 55.681530097743725%.


#### Add Results to Table

In [ ]:
df_summary = pd.read_csv("results/metrics/clustering_summary.csv")

summary_path = "results/metrics/clustering_summary.csv"

new_row = pd.DataFrame([{
    "method": "KMeans Full 28",
    "dataset_version": "HIGGS.csv(full)",
    "n_clusters": 2,
    "n_rows": X_scaled.shape[0],
    "n_features": X_scaled.shape[1],
    "training_time_seconds": training_time,
    "prediction_time_seconds": prediction_time,
    "iterations": kmeans.n_iter_,
    "accuracy": accuracy,
    "silhouette_score": None,
    "davies_bouldin": None,
    "compactness": None,
    "separation": None
}])

if os.path.exists(summary_path):
    df_summary = pd.read_csv(summary_path)
    df_summary = pd.concat([df_summary, new_row], ignore_index=True)
else:
    df_summary = new_row

print(df_summary.tail())

           method     dataset_version  n_clusters    n_rows  n_features  \
1  KMeans Full 28         sample_200k           2    200000          28   
2    KMeans PCA 5         sample_200k           2    200000           5   
3   KMeans PCA 10         sample_200k           2    200000          10   
4    KMeans PCA 2         sample_200k           2    200000           2   
5  KMeans Full 28  HIGGS.csv.gz(full)           2  10721302          28   

   training_time_seconds  prediction_time_seconds  iterations  accuracy  \
1               1.632142                 0.024272           6  55.63850   
2               5.958061                 0.018394          14  55.67800   
3               7.992132                 0.025261           6  55.63800   
4               6.864301                 0.015590          18  55.61950   
5              16.070961                 2.503036          18  55.68153   

   silhouette_score  davies_bouldin  compactness  separation  \
1          0.116011        2.71498

C:\Users\austi\AppData\Local\Temp\ipykernel_12784\1483223608.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_summary = pd.concat([df_summary, new_row], ignore_index=True)


#### Read Sampled Data In

In [ ]:
X = pd.read_csv("data/processed/X_sample.csv")
print("X Shape:", X.shape)
print("\nFirst rows of X:")
print(X.head())


y = pd.read_csv("data/processed/y_sample.csv")
print("y Shape:", y.shape)
print("\nFirst rows of y:")
print(y.head())

X Shape: (200000, 28)

First rows of X:
   lepton_pT  lepton_eta  lepton_phi  missing_E_mag  missing_E_phi   jet1_pT  \
0   1.138683   -0.726635   -0.005790       0.204118       0.153842  1.585904   
1   0.404633    1.014821   -1.050041       1.136441      -1.403536  3.218436   
2   1.137585    0.325251    1.453598       0.804114       0.893516  0.418095   
3   1.380438   -0.595149   -0.727112       0.465392      -0.057453  0.399224   
4   0.962628    1.191110   -1.161568       1.541759       0.569159  1.337374   

   jet1_eta  jet1_phi  jet1_btag   jet2_pT  ...  jet4_eta  jet4_phi  \
0 -0.045576 -1.448527   1.086538  1.598471  ... -2.439800  0.073642   
1 -1.944837  0.801788   0.000000  2.238942  ... -1.174742 -0.912542   
2 -1.164536 -0.585919   0.000000  0.653565  ...  0.280201 -0.982461   
3 -0.076273  1.080084   2.173076  0.644878  ...  1.261267  1.129085   
4  0.810973  0.458075   1.086538  0.549946  ...  0.413452  1.309431   

   jet4_btag      m_jj     m_jjj      m_lv     m_jlv

#### Scale Features

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

#### Fit Model and Predict

In [ ]:
start_time = time.time()

kmeans = KMeans(n_clusters=2, init = 'k-means++', random_state = 42, n_init=20) 

kmeans.fit(X_scaled)

end_time = time.time()

training_time = end_time - start_time
print(f"K-means runtime: {training_time:.2f} seconds")
print(f"Iterations: {kmeans.n_iter_}")

start_time = time.time()

y_kmeans = kmeans.predict(X_scaled)

end_time = time.time()
prediction_time = end_time - start_time

print(f"K-means prediction time: {prediction_time:.2f} seconds")

K-means runtime: 2.45 seconds
Iterations: 6
K-means prediction time: 0.06 seconds


Parameters:
- init = 'k-means++' provides a more robust starting point. Instead of a random centroid initialization, k-means++ first chooses a random centroid from the data points. Then for each subsequent centorid, it selects a data point with a probability proportional to its squared distance from the nearest existing centroid. This process favors points that are further away from chosen centers, leading to a more dispersed and strategic intital placement. It increases the likelihood of finding a better final clustering solution and often reduces the number of iterations needed for convergence. 

- n_clusters=2 since there are two distinct classes, signal and background.

- random_state = 42 for reproducibility.

- n_init=20 for more stability

- max_iter = 300 is the default; convergence happens much earlier (~6 iterations)

#### Evaluate with y labels

Since K-means is unsupervised, the clusters might not correspond with the true label. Since we have the label, the following code below maps the cluster labels to the majority true label inside each cluster. 

In [ ]:
mapped_labels = np.zeros_like(y_kmeans) # same length as y_kmeans

for i in range(2):
    bool_mask = (y_kmeans==i)
    mapped_labels[bool_mask] = mode(y[bool_mask], keepdims=True).mode[0]

accuracy = accuracy_score(y, mapped_labels)*100
print(f"The accuracy of the K-means algorithm on the full feature sampled dataset is {accuracy}%.")


The accuracy of the K-means algorithm on the full feature sampled dataset is 54.7885%.


The K-means clustering achieved an accuracy of ~ 55.64% when the cluster labels were mapped to the majority class within each cluster. This is slightly higher than the baseline accuracy of 54.33%, which was calculated in the `prepare_data.ipynb`. This indicates the clusters that the K-means algorithm formed do not align well with true signal/background labels. This makes sense, as K-means groups observations based on feature similarity rather than class labels. The full feature dataset contains a lot of similar features that overlap, so the accuracy and other metrics should be improved after PCA. 

#### Add Results to clustering_summary.csv

In [ ]:
# Add to results table to compare models
new_row = pd.DataFrame([{
    "method": "KMeans Full 28",
    "dataset_version": "sample_200k",
    "n_clusters": 2,
    "n_rows": X_scaled.shape[0],
    "n_features": X_scaled.shape[1],
    "training_time_seconds": training_time,
    "prediction_time_seconds": prediction_time,
    "iterations": kmeans.n_iter_,
    "accuracy": accuracy,
    "silhouette_score": None,
    "davies_bouldin": None,
    "compactness": None,
    "separation": None
}])


df_summary = pd.concat([df_summary, new_row], ignore_index=True)

df_summary.to_csv(summary_path, index=False)

print(df_summary.tail())

           method     dataset_version  n_clusters    n_rows  n_features  \
2    KMeans PCA 5         sample_200k           2    200000           5   
3   KMeans PCA 10         sample_200k           2    200000          10   
4    KMeans PCA 2         sample_200k           2    200000           2   
5  KMeans Full 28  HIGGS.csv.gz(full)           2  10721302          28   
6  KMeans Full 28         sample_200k           2    200000          28   

   training_time_seconds  prediction_time_seconds  iterations  accuracy  \
2               5.958061                 0.018394          14  55.67800   
3               7.992132                 0.025261           6  55.63800   
4               6.864301                 0.015590          18  55.61950   
5              16.070961                 2.503036          18  55.68153   
6               2.448016                 0.062395           6  54.78850   

   silhouette_score  davies_bouldin  compactness  separation  \
2          0.349828        1.47011

C:\Users\austi\AppData\Local\Temp\ipykernel_12784\3392230845.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_summary = pd.concat([df_summary, new_row], ignore_index=True)
